# MeetingSentiment-PhoBERT-v1

Notebook này fine-tune PhoBERT cho sentiment transcript họp tiếng Việt. Dữ liệu phải được phê duyệt, có consent/license phù hợp, và có ba nhãn `negative`, `neutral`, `positive`.

Notebook không dùng cho phân loại trầm cảm, đánh giá sức khỏe tâm thần, hoặc đánh giá con người. Nó chỉ nhận corpus meeting đã consent có manifest/hash; sẽ dừng nếu thiếu `meeting_id`, leakage giữa split, nhãn/schema không đúng hoặc manifest không khớp.

In [ ]:
# Chạy một lần trên Colab. Runtime GPU T4 hoặc tốt hơn được khuyến nghị.
!pip -q install "transformers>=4.46,<5" "datasets>=3,<4" "accelerate>=1,<2" "scikit-learn>=1.4,<2" "underthesea>=6,<9"

In [ ]:
# Mount Google Drive, sau đó sửa DATA_DIR cho dataset đã được phê duyệt.
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DATA_DIR = Path('/content/drive/MyDrive/meeting-sentiment-approved-v1')
TRAIN_CSV = DATA_DIR / 'train.csv'
VALIDATION_CSV = DATA_DIR / 'validation.csv'
TEST_CSV = DATA_DIR / 'test.csv'
MANIFEST_PATH = DATA_DIR / 'dataset-manifest.json'
OUTPUT_DIR = Path('/content/drive/MyDrive/meeting-sentiment-phobert-v1')

DATASET_VERSION = 'meeting-sentiment-approved-v1'
MODEL_VERSION = 'meeting-sentiment-phobert-v1'
BASE_MODEL = 'vinai/phobert-base-v2'
SEED = 42
MAX_LENGTH = 256
EPOCHS = 4
LEARNING_RATE = 2e-5
BATCH_SIZE = 16
MIN_COVERAGE = 0.80

for split_path in (TRAIN_CSV, VALIDATION_CSV, TEST_CSV, MANIFEST_PATH):
    if not split_path.is_file():
        raise FileNotFoundError(f'Missing approved split: {split_path}')
print('Configured approved dataset:', DATASET_VERSION)

In [ ]:
import hashlib
import json
import random
import shutil
import unicodedata
import zipfile
from collections import Counter
from datetime import UTC, datetime

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
from underthesea import word_tokenize

LABELS = ('negative', 'neutral', 'positive')
LABEL_TO_ID = {label: index for index, label in enumerate(LABELS)}
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
required_manifest_fields = {
    'dataset_version', 'dataset_kind', 'source', 'license_or_consent',
    'collection_period', 'redaction_policy', 'label_guideline_version',
    'annotation_policy', 'split_policy', 'meeting_id_policy', 'output_sha256',
}
missing_manifest_fields = sorted(key for key in required_manifest_fields if not manifest.get(key))
if missing_manifest_fields:
    raise ValueError(f'Manifest is missing required fields: {missing_manifest_fields}')
if manifest['dataset_version'] != DATASET_VERSION:
    raise ValueError('Manifest dataset_version does not match DATASET_VERSION')
if manifest['dataset_kind'] != 'consented-meeting-transcript-sentiment':
    raise ValueError('Notebook only trains the consented meeting-transcript-sentiment profile')
input_paths = {'train': TRAIN_CSV, 'validation': VALIDATION_CSV, 'test': TEST_CSV}
for split_name, split_path in input_paths.items():
    if manifest['output_sha256'].get(split_name) != sha256_file(split_path):
        raise ValueError(f'Manifest SHA-256 does not match {split_name}.csv')

def normalize_text(value):
    return ' '.join(unicodedata.normalize('NFKC', str(value)).strip().split())

def load_split(path):
    frame = pd.read_csv(path, encoding='utf-8-sig', dtype='string')
    required = {'text', 'label', 'meeting_id'}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f'{path.name} is missing required columns: {sorted(missing)}')
    frame = frame.loc[:, ['text', 'label', 'meeting_id']].copy()
    frame['text'] = frame['text'].fillna('').map(normalize_text)
    frame['label'] = frame['label'].fillna('').str.strip().str.lower()
    frame['meeting_id'] = frame['meeting_id'].fillna('').str.strip()
    if (frame['text'] == '').any() or (frame['meeting_id'] == '').any():
        raise ValueError(f'{path.name} contains empty text or meeting_id')
    invalid_labels = sorted(set(frame['label']) - set(LABELS))
    if invalid_labels:
        raise ValueError(f'{path.name} has unsupported labels: {invalid_labels}')
    if frame.empty:
        raise ValueError(f'{path.name} has no rows')
    duplicate_text_rows = int(frame['text'].str.casefold().duplicated().sum())
    if duplicate_text_rows:
        raise ValueError(f'{path.name} has {duplicate_text_rows} duplicate normalized text rows')
    if frame['meeting_id'].str.startswith('proxy-').any():
        raise ValueError(f'{path.name} uses proxy-prefixed meeting IDs')
    return frame

splits = {
    'train': load_split(TRAIN_CSV),
    'validation': load_split(VALIDATION_CSV),
    'test': load_split(TEST_CSV),
}

for left, right in (('train', 'validation'), ('train', 'test'), ('validation', 'test')):
    shared_meetings = set(splits[left]['meeting_id']) & set(splits[right]['meeting_id'])
    if shared_meetings:
        raise ValueError(f'meeting_id leakage between {left} and {right}: {len(shared_meetings)}')
    left_text = set(splits[left]['text'].str.casefold())
    right_text = set(splits[right]['text'].str.casefold())
    if left_text & right_text:
        raise ValueError(f'exact-text leakage between {left} and {right}: {len(left_text & right_text)}')

for name, frame in splits.items():
    missing_labels = set(LABELS) - set(frame['label'])
    if missing_labels:
        raise ValueError(f'{name} is missing labels: {sorted(missing_labels)}')
    print(name, {'units': len(frame), 'meetings': frame['meeting_id'].nunique(), 'labels': frame['label'].value_counts().to_dict()})

quality_report = {
    'schema_version': 1,
    'dataset_profile': 'approved-meeting',
    'dataset_version': DATASET_VERSION,
    'manifest_sha256': sha256_file(MANIFEST_PATH),
    'input_sha256': {name: sha256_file(path) for name, path in input_paths.items()},
    'ready_to_train': True,
    'splits': {
        name: {
            'rows': int(len(frame)),
            'unique_meeting_ids': int(frame['meeting_id'].nunique()),
            'unique_texts': int(frame['text'].str.casefold().nunique()),
            'label_counts': {label: int((frame['label'] == label).sum()) for label in LABELS},
        }
        for name, frame in splits.items()
    },
}


In [ ]:
def segment_vi(text):
    return word_tokenize(text, format='text')

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=False)

def make_dataset(frame):
    dataset = Dataset.from_dict({
        'text': [segment_vi(value) for value in frame['text'].tolist()],
        'labels': [LABEL_TO_ID[value] for value in frame['label'].tolist()],
    })
    return dataset.map(
        lambda batch: tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH),
        batched=True,
        remove_columns=['text'],
    )

encoded = {name: make_dataset(frame) for name, frame in splits.items()}
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(LABELS),
    id2label={index: label for index, label in enumerate(LABELS)},
    label2id=LABEL_TO_ID,
)

def compute_metrics(prediction):
    logits, labels = prediction
    predicted = np.argmax(logits, axis=1)
    return {
        'macro_f1': float(f1_score(labels, predicted, labels=range(len(LABELS)), average='macro', zero_division=0)),
        'accuracy': float(accuracy_score(labels, predicted)),
    }

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'checkpoints'),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    seed=SEED,
    report_to='none',
    logging_strategy='steps',
    logging_steps=25,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded['train'],
    eval_dataset=encoded['validation'],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)
trainer.train()

In [ ]:
def softmax(logits, temperature=1.0):
    scaled = logits / temperature
    scaled = scaled - scaled.max(axis=1, keepdims=True)
    exponent = np.exp(scaled)
    return exponent / exponent.sum(axis=1, keepdims=True)

def fit_temperature(logits, labels):
    candidates = np.linspace(0.5, 3.0, 101)
    losses = []
    for temperature in candidates:
        probabilities = softmax(logits, temperature)
        losses.append(float(-np.log(np.clip(probabilities[np.arange(len(labels)), labels], 1e-12, 1.0)).mean()))
    return float(candidates[int(np.argmin(losses))])

def choose_threshold(probabilities, labels):
    predicted = probabilities.argmax(axis=1)
    confidence = probabilities.max(axis=1)
    best = None
    for threshold in np.linspace(1 / len(LABELS), 0.95, 63):
        accepted = confidence >= threshold
        coverage = float(accepted.mean())
        if coverage < MIN_COVERAGE or not accepted.any():
            continue
        accepted_f1 = float(f1_score(labels[accepted], predicted[accepted], labels=range(len(LABELS)), average='macro', zero_division=0))
        candidate = {'threshold': float(threshold), 'coverage': coverage, 'accepted_macro_f1': accepted_f1}
        if best is None or (accepted_f1, coverage) > (best['accepted_macro_f1'], best['coverage']):
            best = candidate
    if best is None:
        raise ValueError('No threshold reaches MIN_COVERAGE on validation.')
    return best

def report_metrics(labels, probabilities, threshold):
    predicted = probabilities.argmax(axis=1)
    accepted = probabilities.max(axis=1) >= threshold
    precision, recall, f1, support = precision_recall_fscore_support(labels, predicted, labels=range(len(LABELS)), zero_division=0)
    result = {
        'accuracy': float(accuracy_score(labels, predicted)),
        'macro_f1': float(f1_score(labels, predicted, labels=range(len(LABELS)), average='macro', zero_division=0)),
        'coverage': float(accepted.mean()),
        'unknown_rate': float((~accepted).mean()),
        'confusion_matrix': confusion_matrix(labels, predicted, labels=range(len(LABELS))).tolist(),
        'per_label': {
            label: {'precision': float(precision[i]), 'recall': float(recall[i]), 'f1': float(f1[i]), 'support': int(support[i])}
            for i, label in enumerate(LABELS)
        },
    }
    if accepted.any():
        result['accepted_macro_f1'] = float(f1_score(labels[accepted], predicted[accepted], labels=range(len(LABELS)), average='macro', zero_division=0))
    return result

validation_prediction = trainer.predict(encoded['validation'])
validation_logits = np.asarray(validation_prediction.predictions)
validation_labels = np.asarray(validation_prediction.label_ids)
temperature = fit_temperature(validation_logits, validation_labels)
threshold_selection = choose_threshold(softmax(validation_logits, temperature), validation_labels)

test_prediction = trainer.predict(encoded['test'])
test_logits = np.asarray(test_prediction.predictions)
test_labels = np.asarray(test_prediction.label_ids)
evaluation = {
    'schema_version': 1,
    'model_type': 'phobert-sequence-classification',
    'model_version': MODEL_VERSION,
    'dataset_version': DATASET_VERSION,
    'dataset_profile': 'approved-meeting',
    'dataset_manifest_sha256': quality_report['manifest_sha256'],
    'labels': list(LABELS),
    'temperature': temperature,
    'unknown_threshold': threshold_selection['threshold'],
    'threshold_selection_validation': threshold_selection,
    'validation': report_metrics(validation_labels, softmax(validation_logits, temperature), threshold_selection['threshold']),
    'test': report_metrics(test_labels, softmax(test_logits, temperature), threshold_selection['threshold']),
    'split_counts': {name: {'units': len(frame), 'meetings': int(frame['meeting_id'].nunique())} for name, frame in splits.items()},
    'hyperparameters': {'base_model': BASE_MODEL, 'max_length': MAX_LENGTH, 'epochs': EPOCHS, 'learning_rate': LEARNING_RATE, 'batch_size': BATCH_SIZE, 'seed': SEED},
    'created_at_utc': datetime.now(UTC).isoformat(),
}

if OUTPUT_DIR.exists():
    raise FileExistsError(f'Choose a new OUTPUT_DIR; refusing to overwrite: {OUTPUT_DIR}')
(OUTPUT_DIR / 'model').mkdir(parents=True)
(OUTPUT_DIR / 'quality-report.json').write_text(json.dumps(quality_report, ensure_ascii=False, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'dataset-manifest.json').write_bytes(MANIFEST_PATH.read_bytes())
trainer.save_model(str(OUTPUT_DIR / 'model'))
tokenizer.save_pretrained(str(OUTPUT_DIR / 'model'))
(OUTPUT_DIR / 'evaluation.json').write_text(json.dumps(evaluation, ensure_ascii=False, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'label_mapping.json').write_text(json.dumps(LABEL_TO_ID, ensure_ascii=False, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'MODEL_CARD.md').write_text(
    f'# {MODEL_VERSION}\n\n'
    f'- Dataset version: `{DATASET_VERSION}`\n'
    f'- Dataset manifest SHA-256: `{quality_report["manifest_sha256"]}`\n'
    '- Task: three-class polarity of consented Vietnamese meeting transcript units.\n'
    '- Labels: `negative`, `neutral`, `positive`; low-confidence output is `unknown`.\n'
    '- Not for depression screening, clinical assessment, or decisions about people.\n',
    encoding='utf-8',
)

archive_path = shutil.make_archive(str(OUTPUT_DIR), 'zip', root_dir=OUTPUT_DIR)
print(json.dumps({'artifact_zip': archive_path, 'test': evaluation['test']}, ensure_ascii=False, indent=2))

## Sau khi chạy

Lưu `evaluation.json`, `MODEL_CARD.md`, dataset manifest và archive model vào evidence. Chỉ cân nhắc tích hợp artifact sau khi test set theo meeting đạt tiêu chí đã chốt. Backend hiện chưa có adapter serving PhoBERT; không thay `baseline-rules-v2` bằng file model này nếu chưa bổ sung adapter và test contract.